# Notebook 05 - SPY and VIX Raw Data Validation

This notebook documents Stage 6 of the revised project. SPY replaces the non-traded S&P 500 index (`^GSPC`) because SPY has directly traded share volume. VIX remains outside the main HMM because it is reserved as an external validator and later regression control.

This stage downloads or reloads raw SPY and VIX checkpoints, validates them, and aligns their common trading dates. It does **not** create returns, abnormal volume, drawdown, GARCH estimates, regimes, transition outcomes, or statistical models.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'configs' / 'research_config.yaml').exists():
    raise FileNotFoundError('Run this notebook from the notebooks directory.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from market_regime.config import load_research_config
from market_regime.data import (
    compare_trading_calendars,
    create_aligned_raw_market_data,
    download_yahoo_ticker,
    prepare_raw_ticker_data,
    save_csv_checkpoint,
    validate_spy_data,
    validate_vix_data,
)


In [2]:
config = load_research_config(PROJECT_ROOT / 'configs' / 'research_config.yaml')
data_config = config['data']
raw_dir = PROJECT_ROOT / config['storage']['raw_data_directory']
processed_dir = PROJECT_ROOT / config['storage']['processed_data_directory']
spy_path = raw_dir / 'spy_daily.csv'
vix_path = raw_dir / 'vix_daily.csv'
aligned_path = processed_dir / 'spy_vix_aligned_raw.csv'

print('Configured market ticker:', data_config['market_ticker'])
print('Configured VIX ticker:', data_config['vix_ticker'])
print('Research dates:', data_config['start_date'], 'through', data_config['end_date'])


Configured market ticker: SPY
Configured VIX ticker: ^VIX
Research dates: 2000-01-01 through 2026-05-31


## Checkpoint workflow

The cell below reloads existing Stage 6 checkpoints when all three files are present. If they are absent, it downloads the configured Yahoo Finance tickers with `auto_adjust=False`, preserves raw OHLC data and Adjusted Close when supplied, validates both datasets, aligns common dates, and writes the same checkpoints.

In [3]:
checkpoint_paths = (spy_path, vix_path, aligned_path)
if all(path.exists() for path in checkpoint_paths):
    print('Loading existing Stage 6 checkpoints.')
    spy = pd.read_csv(spy_path, index_col='Date', parse_dates=True)
    vix = pd.read_csv(vix_path, index_col='Date', parse_dates=True)
else:
    print('Downloading SPY and ^VIX from Yahoo Finance.')
    start, end = data_config['start_date'], data_config['end_date']
    spy_download = download_yahoo_ticker(data_config['market_ticker'], start, end)
    vix_download = download_yahoo_ticker(data_config['vix_ticker'], start, end)
    spy = prepare_raw_ticker_data(spy_download, data_config['market_ticker'], start, end)
    vix = prepare_raw_ticker_data(vix_download, data_config['vix_ticker'], start, end, vix_prefix=True)

spy.index.name = 'Date'
vix.index.name = 'Date'
spy_report = validate_spy_data(spy, data_config['start_date'], data_config['end_date'])
vix_report = validate_vix_data(vix, data_config['start_date'], data_config['end_date'])
calendar_report = compare_trading_calendars(spy, vix)
aligned = create_aligned_raw_market_data(spy, vix)

save_csv_checkpoint(spy, spy_path)
save_csv_checkpoint(vix, vix_path)
save_csv_checkpoint(aligned, aligned_path)
print('Stage 6 checkpoints saved.')


Loading existing Stage 6 checkpoints.


Stage 6 checkpoints saved.


In [4]:
for name, frame, report in [('SPY', spy, spy_report), ('VIX', vix, vix_report)]:
    print(f'\n{name}: {report["row_count"]:,} rows from {report["first_available_date"]} through {report["last_available_date"]}')
    print('Data types:')
    display(frame.dtypes.to_frame('dtype'))
    print('Missing values:')
    display(pd.Series(report['missing_values'], name='missing_values'))
    display(frame.head())
    display(frame.tail())

print('SPY volume-quality checks:')
display(pd.Series({
    'zero_volume_observations': spy_report['zero_volume_observations'],
    'negative_volume_observations': spy_report['negative_volume_observations'],
    'nonfinite_volume_observations': spy_report['nonfinite_values']['Volume'],
}, name='count'))



SPY: 6,641 rows from 2000-01-03 through 2026-05-29
Data types:


,dtype
Adj_Close,float64
Close,float64
High,float64
Low,float64
Open,float64
Volume,int64


Missing values:


Adj_Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
Name: missing_values, dtype: int64

,Adj_Close,Close,High,Low,Open,Volume
Date,,,,,,
2000-01-03,91.132751,145.4375,148.25000,143.875000,148.25000,8164300
2000-01-04,87.568909,139.7500,144.06250,139.640625,143.53125,8089800
2000-01-05,87.725540,140.0000,141.53125,137.250000,139.93750,12177900
2000-01-06,86.315666,137.7500,141.50000,137.750000,139.62500,6227200
2000-01-07,91.328560,145.7500,145.75000,140.062500,140.31250,8066500


...

VIX_Adj_Close    0
VIX_Close        0
VIX_High         0
VIX_Low          0
VIX_Open         0
VIX_Volume       0
Name: missing_values, dtype: int64

,VIX_Adj_Close,VIX_Close,VIX_High,VIX_Low,VIX_Open,VIX_Volume
Date,,,,,,
2000-01-03,24.209999,24.209999,26.150000,23.980000,24.360001,0
2000-01-04,27.010000,27.010000,27.180000,24.799999,24.940001,0
2000-01-05,26.410000,26.410000,29.000000,25.850000,27.980000,0
2000-01-06,25.730000,25.730000,26.709999,24.700001,26.680000,0
2000-01-07,21.719999,21.719999,25.170000,21.719999,25.139999,0


,VIX_Adj_Close,VIX_Close,VIX_High,VIX_Low,VIX_Open,VIX_Volume
Date,,,,,,
2026-05-25,16.590000,16.590000,16.870001,16.530001,16.809999,0
2026-05-26,17.010000,17.010000,17.230000,16.559999,16.920000,0
2026-05-27,16.290001,16.290001,17.180000,16.290001,17.010000,0
2026-05-28,15.740000,15.740000,16.850000,15.610000,16.760000,0
2026-05-29,15.320000,15.320000,15.880000,15.220000,15.810000,0


SPY volume-quality checks:


zero_volume_observations         0
negative_volume_observations     0
nonfinite_volume_observations    0
Name: count, dtype: int64

In [5]:
print('Calendar alignment report:')
display(pd.Series({
    'common_dates': calendar_report['common_count'],
    'SPY_only_dates': calendar_report['spy_only_count'],
    'VIX_only_dates': calendar_report['vix_only_count'],
}))
print('SPY-only examples:', calendar_report['spy_only_examples'])
print('VIX-only examples:', calendar_report['vix_only_examples'])
print(f'Aligned raw dataset: {len(aligned):,} common trading dates')
display(aligned.head())
display(aligned.tail())


Calendar alignment report:


common_dates      6641
SPY_only_dates       0
VIX_only_dates       1
dtype: int64

SPY-only examples: []
VIX-only examples: ['2026-05-25']
Aligned raw dataset: 6,641 common trading dates


,Adj_Close,Close,High,Low,Open,Volume,VIX_Adj_Close,VIX_Close,VIX_High,VIX_Low,VIX_Open,VIX_Volume
Date,,,,,,,,,,,,
2000-01-03,91.132751,145.4375,148.25000,143.875000,148.25000,8164300,24.209999,24.209999,26.150000,23.980000,24.360001,0
2000-01-04,87.568909,139.7500,144.06250,139.640625,143.53125,8089800,27.010000,27.010000,27.180000,24.799999,24.940001,0
2000-01-05,87.725540,140.0000,141.53125,137.250000,139.93750,12177900,26.410000,26.410000,29.000000,25.850000,27.980000,0
2000-01-06,86.315666,137.7500,141.50000,137.750000,139.62500,6227200,25.730000,25.730000,26.709999,24.700001,26.680000,0
2000-01-07,91.328560,145.7500,145.75000,140.062500,140.31250,8066500,21.719999,21.719999,25.170000,21.719999,25.139999,0


,Adj_Close,Close,High,Low,Open,Volume,VIX_Adj_Close,VIX_Close,VIX_High,VIX_Low,VIX_Open,VIX_Volume
Date,,,,,,,,,,,,
2026-05-22,743.723999,745.640015,748.940002,744.479980,746.239990,41762000,16.700001,16.700001,17.389999,16.459999,16.83,0
2026-05-26,748.661316,750.590027,752.130005,748.369995,750.010010,41123600,17.010000,17.010000,17.230000,16.559999,16.92,0
2026-05-27,748.531616,750.460022,751.380005,748.219971,750.880005,42106300,16.290001,16.290001,17.180000,16.290001,17.01,0
2026-05-28,752.660950,754.599976,755.150024,749.229980,750.250000,41562600,15.740000,15.740000,16.850000,15.610000,16.76,0
2026-05-29,754.536133,756.479980,758.080017,754.690002,755.900024,55075700,15.320000,15.320000,15.880000,15.220000,15.81,0


## Stage 6 boundary

The three CSV files are generated raw-data checkpoints. At this point, the project has validated raw SPY and VIX observations and their date alignment only. Feature construction and all model estimation remain future stages.